In [3]:
%env AWS_PROFILE=platform-developer
%load_ext autoreload
%autoreload 2

env: AWS_PROFILE=platform-developer


In [4]:
import os
from pathlib import Path
from lxml import etree
import json

SOURCE_WORKS_ROOT = "/Users/brychtas/Documents/all-source-works"
CALM_RAW_WORKS_ROOT = "/Users/brychtas/Documents/all-raw-calm-works"
AXIELL_RAW_WORKS_ROOT = "/Users/brychtas/Documents/all-raw-axiell-works"

def load_all_calm_works():
    calm_file_names = [f.name for f in Path(CALM_RAW_WORKS_ROOT).iterdir() if f.is_file()]
    
    calm_works = {}
    for file_name in calm_file_names:
        with open(os.path.join(CALM_RAW_WORKS_ROOT, file_name)) as f:
            parsed_work = json.load(f)
            calm_works[parsed_work["id"]] = parsed_work

    return calm_works


def load_all_axiell_works():
    axiell_file_names = [f.name for f in Path(AXIELL_RAW_WORKS_ROOT).iterdir() if f.is_file()]
    axiell_works = {}

    for file_name in axiell_file_names:
        if not file_name.endswith(".xml"):
            continue

        parsed_work = etree.parse(os.path.join(AXIELL_RAW_WORKS_ROOT, file_name))
        identifier = file_name.split(".")[0].replace("_", ":")
        axiell_works[identifier] = parsed_work

    return axiell_works


def load_all_calm_source_works():
    all_calm_source_works = {}

    calm_source_work_file_names = [f.name for f in Path(f"{SOURCE_WORKS_ROOT}/calm-record-id").iterdir() if f.is_file()]
    
    for file_name in calm_source_work_file_names:
        if not file_name.endswith(".json"):
            continue
        
        with open(f"{SOURCE_WORKS_ROOT}/calm-record-id/{file_name}") as f:
            all_calm_source_works[file_name] = json.load(f)

    return all_calm_source_works

In [33]:
# calm_works = load_all_calm_works()
# print(len(calm_works))

axiell_works = load_all_axiell_works()
print(len(axiell_works))

# calm_source_works = load_all_calm_source_works()
# print(len(calm_source_works))

# axiell_marc_works = axiell_oai_to_marc()
# print(len(axiell_marc_works))

227789


In [31]:
from collections import Counter, defaultdict

def find_xml_value(record, field_path: list[str]):
    ns = {"oai": "http://www.openarchives.org/OAI/2.0/"}
    
    path = "/".join([f"oai:{val}" for val in field_path])
    element = record.find(path, namespaces=ns)
    if element is not None:
        return element.text


def count_oai_values(field_path: list[str]):
    values = []
    value_identifier_mapping = defaultdict(list)
    
    for identifier, record in axiell_works.items():
        value = find_xml_value(record, field_path)
        values.append(value)
        value_identifier_mapping[value].append(identifier)

    return Counter(values), value_identifier_mapping


def map_axiell_ids_to_calm_ids():
    mapping = {}
    for axiell_id, record in axiell_works.items():
        calm_id = find_xml_value(record, ["PIDother", "PID_other.non-URI_ID"])
        mapping[axiell_id] = calm_id

    return mapping

In [34]:
axiell_to_calm = map_axiell_ids_to_calm_ids()

In [ ]:
counter, mapping = count_oai_values(["accession_status", "value"])
print(counter, len(counter))
for i, axiell_id in enumerate(mapping["PRIVATE"]):
    print(axiell_id, axiell_to_calm[axiell_id], calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("AccessStatus"))
    if i > 100:
        break

In [ ]:
counter, mapping = count_oai_values(["Record_progress", "record_progress", "value[@lang='0']"])
print(counter)
for i, axiell_id in enumerate(mapping["draft"]):
    print(axiell_to_calm[axiell_id], calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("CatalogueStatus"))
    if i > 100:
        break

In [ ]:
counter, mapping = count_oai_values(["Inscription", "inscription.language"])
print(counter.keys())



In [ ]:
counter, mapping = count_oai_values(["record_type", "value[@lang='0']"])
print(counter)
for i, axiell_id in enumerate(mapping["Work"]):
    print(axiell_id, axiell_to_calm[axiell_id], calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("Level"))
    if i > 10:
        break

In [ ]:
counter, mapping = count_oai_values(["Object_category", "object_category"])
print(counter)

In [ ]:
# All Mimsy works have consecutive identifiers < 20,000.
# No record with consecutive identifier < 20,000 has a predecessor identifier.

mimsy_count = 0
calm_count = 0
missing_pid_count = 0

for identifier, record in axiell_works.items():
    pid_value = find_xml_value(record, ["PIDother", "PID_other.non-URI_ID"])
    mimsy_value = find_xml_value(record, ["Description", "description.note"])


    
    numeric_id = int(identifier.split(":")[1])
    
    if mimsy_value:
        mimsy_count += 1
    if pid_value:
        calm_count += 1

    if numeric_id > 20000 and not (value := find_xml_value(record, ["current_location.context"])) and find_xml_value(record, ["current_location.name"]):
        print("!!!!!!!!!!")
        print(value)
        break
        #print(etree.tostring(record, pretty_print=True, encoding="unicode"))
        
            
    if not pid_value and numeric_id > 20000:
        #print(etree.tostring(record, pretty_print=True, encoding="unicode"))
        missing_pid_count += 1

print(mimsy_count)
print(calm_count)
print(missing_pid_count)

In [ ]:
%pip install deepdiff

In [7]:
from utils.elasticsearch import get_client
from core.source import ElasticSource
import json
import os

es = get_client("read_only", pipeline_date="2025-10-02", es_mode="public")
source = ElasticSource(es, "works-source-2025-10-02", query={"match_all": {}})

for work in source.stream_raw():
    work_id = work['state']['sourceIdentifier']['value'].replace("/", "+")
    work_id_type = work['state']['sourceIdentifier']['identifierType']['id']

    os.makedirs(f"{SOURCE_WORKS_ROOT}/{work_id_type}", exist_ok=True)
    try:
        with open(f"{SOURCE_WORKS_ROOT}/{work_id_type}/{work_id}.json", "w") as f:
            f.write(json.dumps(work))
    except Exception as e:
        print(e)

In [26]:
print(len(all_calm_source_works))
print("Non-deleted works", len([w for w in all_calm_source_works.items() if "data" in w[1]]))

works_with_terms_of_use_note = 0
terms_of_use_contents = []
for work_id, work in all_calm_source_works.items():
    # Skip deleted works
    if "data" not in work:
        continue

    for note in work["data"]["notes"]:
        if note["noteType"]["id"] == "terms-of-use":
            works_with_terms_of_use_note += 1
            terms_of_use_contents.append(note["contents"])
            #print(note["contents"])

print(len(terms_of_use_contents))

408079
Non-deleted works 256530
83418


In [33]:
from collections import Counter
print(len(set(terms_of_use_contents)))

for val in list(Counter(terms_of_use_contents))[:100]:
    print(val)


1890
This item is restricted. You will need to complete a Restricted Access form agreeing to anonymise personal data before you will be allowed to view this item. Please see our <a href="https://wellcomecollection.cdn.prismic.io/wellcomecollection/d4817da5-c71a-4151-81c4-83e39ad4f5b3_Wellcome+Collection_Access+Policy_Aug+2020.pdf">Access Policy</a> for more details. Restricted until 1 January 2038.
It was previously closed until 01/01/2024. It was reassessed in January 2024 and subsequently opened.
Please consult the digitised version as this item is fragile. Email library@wellcomecollection.org to request access to the physical item.
Open and available at Cold Spring Harbor Laboratory Library and Archives.
The papers are available at UCL Special Collections and Archives subject to the usual conditions of access to Archives and Manuscripts material, after the completion of a Reader's Undertaking.
This item is not available online. Email collections@wellcomecollection.org to request an 

In [39]:
calm_works["7010beb3-0962-4629-9b30-3e988d3277ae"]

{'id': '7010beb3-0962-4629-9b30-3e988d3277ae',
 'data': {'Level': ['Item'],
  'Location': ['183;LG2.5;FR;19;4'],
  'Created': ['20/12/2021'],
  'UserWrapped6': ['sheet film negative; 16.3 x 11.8 cm'],
  'CountryCode': ['GB'],
  'UserText3': ['Archives - Not Requestable'],
  'Title': ['M0007173: Manuscript illustration of Christ baptising at healing springs'],
  'ACCESS': [''],
  'OtherNumber': ['M0007173'],
  'CONTENT': [''],
  'Inscription': ['H.M.E. C/1270'],
  'Subject': ['<p>Wells and Springs\n'],
  'Transmission': ['Yes'],
  'AccessStatus': ['Open'],
  'Material': ['Archives - Non-digital'],
  'Description': ['Photograph of an unidentified 15th century manuscript page showing a drawing, possibly of Christ baptising at a healing springs. Another photograph of this image was accessioned by the Wellcome Historical Medical Museum on 30th August 1935 (PHO 4227).'],
  'Digitised': ['No'],
  'Modifier': ['Sloyanv'],
  'Copyright': [''],
  'UserDate1': [''],
  'ClosedUntil': [''],
  'Orde

In [54]:
calm_works["9946f428-9d01-43f3-94f3-951a386a0c58"]

{'id': '9946f428-9d01-43f3-94f3-951a386a0c58',
 'data': {'Level': ['Item'],
  'Location': ['183;LG2.5;MR;79;18'],
  'PreviousNumbers': [''],
  'Created': ['29/01/2025'],
  'UserWrapped6': ['watercolour on paper'],
  'CountryCode': ['GB'],
  'Wheels': ['', ''],
  'UserText3': ['Visual Material - Requestable'],
  'Title': ['Two paths, one red and one yellow, with trees and a cat'],
  'ConditionNote': [''],
  'ACCESS': [''],
  'OtherNumber': ['Adamson Trust Inventory Number: 295.12'],
  'Arrangement': ['Prior to Wellcome Collection, this work was stored in a folder on 1st floor, Reay House, Lambeth Hospital.'],
  'CONTENT': [''],
  'Inscription': ['', '2/ 11.1.89'],
  'Subject': ['Cats', 'Trees', 'Trails', 'Landscapes', ''],
  'Transmission': ['Yes'],
  'AccessStatus': ['Open'],
  'Dimensions': ['sheet 30.4 x 42.8 cm'],
  'Former_Location': ['D79.13'],
  'CreatorName': ['Meredith, David Thomas, active approximately 1975-1989'],
  'Materials': ['Paper', ''],
  'Material': ['Visual Material

In [5]:
calm_works = load_all_calm_works()
print(len(calm_works))


408029


In [47]:
from collections import defaultdict

calm_langs = []
mapping = defaultdict(list)

for work_id, work in calm_works.items():
    if 'Date' in work['data'] and work['data']['Date'] != ['']:
        calm_langs += work['data']['Date']
        for lang in work['data']['Date']:
            mapping[lang].append(work_id)
        
        

In [50]:
calm_works["0b14efdd-f012-4864-ab45-8a1b69916bc3"]

{'id': '0b14efdd-f012-4864-ab45-8a1b69916bc3',
 'data': {'UserText2': ['September 2010'],
  'Link_To_Digitised': [''],
  'Appraisal': [''],
  'Level': ['Item'],
  'Location': [''],
  'Format': [''],
  'VehicleDetails': [''],
  'UserText7': [''],
  'Created': ['20/09/2010'],
  'UserWrapped6': ['Format of original recording: wav 44.1 khz 16 bit ZOOM digital recorder.'],
  'CountryCode': ['GB'],
  'Engine': ['Yes'],
  'Wheels': [''],
  'UserText3': [''],
  'Title': ['Baker, Harriet'],
  'ACCESS': [''],
  'OtherNumber': ['One&Other1724'],
  'Arrangement': [''],
  'CustodialHistory': [''],
  'CONTENT': [''],
  'PackingNote': [''],
  'Transmission': ['Yes'],
  'AccessStatus': ['Open'],
  'TargetAudience': [''],
  'Condition': [''],
  'CreatorName': [''],
  'CONSERVATIONREQUIRED': [''],
  'Format_Details': [''],
  'UserWrapped2': [''],
  'UserText9': [''],
  'Accruals': [''],
  'Material': ['Audio-Visual Material - e-sound only'],
  'Acquisition': [''],
  'UserText8': [''],
  'AccessCategory'

In [46]:
for val in set(calm_langs):
    if "Partly in German, partly in English." in val:
        for calm in mapping[val]:
            print(calm, calm_to_axiell.get(calm))

2defb47f-d4c2-4dcd-a3c6-170939a8f3d9 collect:110024245
85941ac3-7376-4f55-b3de-83cc7c4d59dc collect:110118220
3262eb61-ed2c-4d24-8eab-288d627f0b23 collect:110081553
fa97d7dc-6ca9-401e-ac0f-05cb4a4b1824 collect:110047017
e97902f7-3130-4b28-845f-7d4fbc2c0c5a collect:110036578
05b2fbe3-3eb8-46ef-b880-8a08e1173315 collect:110058275
57e5e1f7-065c-4d66-8a3d-c71abf852740 collect:110164059
46a2f529-6a1e-4722-9743-92926812e53b collect:110052589
ff63c0a7-762e-4654-883d-f4b6ed7dd1c9 collect:110046011
b0e2f81d-88d1-4f25-9ef9-b095634e0c62 collect:110061636


In [41]:
for item in axiell_to_calm:
    print(item)
    break

calm_to_axiell = dict()
for k, v in axiell_to_calm.items():
    calm_to_axiell[v] = k

collect:110163064


In [43]:
for item in calm_to_axiell:
    print(item)
    break

f41de44d-64c5-4a0e-bb21-a003d89d8612


In [51]:
calm_to_axiell["f9f09f42-675d-4d27-8efa-1726d314f20b"]

'collect:110159658'

In [65]:
axiell_id = "collect:110085293"
calm_id = axiell_to_calm[axiell_id]
print(find_xml_value(axiell_works[axiell_id], ["Dating", "dating.date.start"]))
print(find_xml_value(axiell_works[axiell_id], ["Dating", "dating.date.end"]))
print(find_xml_value(axiell_works[axiell_id], ["production_date_text"]))
print(calm_works[calm_id]["data"]["Date"])

1955-09-01
1955-09-30
Sep 1955
['Sep 1955']


In [2]:
marc_axiell_works["collect:110169561"]

NameError: name 'marc_axiell_works' is not defined